In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_legal_process_step_qc(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "LegalProcessStepId	string,"
        #     "SourceSystem	string,"
        #     "PortfolioKey	string,"
        #     "PortfolioName	string,"
        #     "IngestionTimestamp	timestamp,"
        #     "BatchId	    string,"
        #     "LegalProcessStepKey	string,"
        #     "LegalProcessKey	int,"
        #     "LegalStageKey int,"
        #     "LegalStepKey int,"
        #     "LegalActionKey int,"
        #     "ExposureKey string,"
        #     "PartyKey int,"
        #     "CDI string,"
        #     "CaseCode int,"
        #     "LegalStepCreatedBy string,"
        #     "LegalStepCreatedDate timestamp,"
        #     "LegalStepStartDate string,"
        #     "LegalStepEndDate string,"
        #     "LegalStepSystemEndDate string,"
        #     "LegalStepDefinitionCode int,"
        #     "LegalStepDefinition string,"
        #     "LegalStepStatusCode int,"
        #     "LegalStepStatus string,"
        #     "LegalStepComment string,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.LegalProcessStep"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qc",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== case_master ==================
        case_master_extra_filters = None

        if case_master_extra_filters is not None:
            case_master_extra_filters.append(date_filter)
            case_master_final_filters = " AND ".join(case_master_extra_filters)
        else:
            case_master_final_filters = date_filter

        case_master_selected_fields = [
            "debt_code",
            "cust_code",
            "case_code",
            "vend_code",
            "old_case_code",
        ]

        case_master = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_case_master",
            case_master_final_filters,
            case_master_selected_fields,
        )

        # ================== customer ==================
        customer_extra_filters = None

        if customer_extra_filters is not None:
            customer_extra_filters.append(date_filter)
            customer_final_filters = " AND ".join(customer_extra_filters)
        else:
            customer_final_filters = date_filter

        customer_selected_fields = [
            "CDI",
            "cust_code",
        ]

        customer = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_customer",
            customer_final_filters,
            customer_selected_fields,
        )

        # ================== Portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "vendor_code_qualco",
        ]

        portfolio = load_landed_portfolio_table(
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== lst_of_val ==================
        lst_of_val_extra_filters = None

        if lst_of_val_extra_filters is not None:
            lst_of_val_extra_filters.append(date_filter)
            lst_of_val_final_filters = " AND ".join(lst_of_val_extra_filters)
        else:
            lst_of_val_final_filters = date_filter

        lst_of_val_selected_fields = [
            "lov_desc",
            "lov_low",
            "lovt_code",
        ]

        lst_of_val = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_lst_of_val",
            lst_of_val_final_filters,
            lst_of_val_selected_fields,
        )

        # ================== process_definition_steps ==================
        process_definition_steps_extra_filters = None

        if process_definition_steps_extra_filters is not None:
            process_definition_steps_extra_filters.append(date_filter)
            process_definition_steps_final_filters = " AND ".join(
                process_definition_steps_extra_filters
            )
        else:
            process_definition_steps_final_filters = date_filter

        process_definition_steps_selected_fields = [
            "pdsp_name",
            "pdsp_code",
        ]

        process_definition_steps = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_process_definition_steps",
            process_definition_steps_final_filters,
            process_definition_steps_selected_fields,
        )

        # ================== process_instance_mapping_activities ==================
        process_instance_mapping_activities_extra_filters = None

        if process_instance_mapping_activities_extra_filters is not None:
            process_instance_mapping_activities_extra_filters.append(date_filter)
            process_instance_mapping_activities_final_filters = " AND ".join(
                process_instance_mapping_activities_extra_filters
            )
        else:
            process_instance_mapping_activities_final_filters = date_filter

        process_instance_mapping_activities_selected_fields = [
            "cac_code",
            "prst_code",
            "case_code",
        ]

        process_instance_mapping_activities = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_process_instance_mapping_activities",
            process_instance_mapping_activities_final_filters,
            process_instance_mapping_activities_selected_fields,
        )

        # ================== process_instance_participants ==================
        process_instance_participants_extra_filters = None

        if process_instance_participants_extra_filters is not None:
            process_instance_participants_extra_filters.append(date_filter)
            process_instance_participants_final_filters = " AND ".join(
                process_instance_participants_extra_filters
            )
        else:
            process_instance_participants_final_filters = date_filter

        process_instance_participants_selected_fields = [
            "case_code",
            "prin_code",
            "cust_code",
        ]

        process_instance_participants = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_process_instance_participants",
            process_instance_participants_final_filters,
            process_instance_participants_selected_fields,
        )

        # ================== process_instance_steps ==================
        process_instance_steps_extra_filters = None

        if process_instance_steps_extra_filters is not None:
            process_instance_steps_extra_filters.append(date_filter)
            process_instance_steps_final_filters = " AND ".join(
                process_instance_steps_extra_filters
            )
        else:
            process_instance_steps_final_filters = date_filter

        process_instance_steps_selected_fields = [
            "prin_code",
            "pris_code",
            "pips_code",
            "prst_code",
            "prst_created_by",
            "prst_created",
            "prst_start_date",
            "prst_completion_date",
            "pdsp_code",
            "prst_status",
            "prst_comments",
            "businessdate",
        ]

        process_instance_steps = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_process_instance_steps",
            process_instance_steps_final_filters,
            process_instance_steps_selected_fields,
        )
        # ======= process_instances ====================

        process_instances_extra_filters = None

        if process_instances_extra_filters is not None:
            process_instances_extra_filters.append(date_filter)
            process_instances_final_filters = " AND ".join(
                process_instances_extra_filters
            )
        else:
            process_instances_final_filters = date_filter

        process_instances_selected_fields = [
            "prin_code",
            "cust_code",
        ]

        process_instances = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_process_instances",
            process_instances_final_filters,
            process_instances_selected_fields,
        )
        
        # ======= process_instance_participants_distinct =======

        process_instance_participants_distinct = process_instance_participants.select(
            process_instance_participants.case_code,
        ).distinct()

        # ================== Join ==================

        # Conditions
        process_instance_steps_process_instances_join_condition = (
            process_instance_steps.prin_code == process_instances.prin_code
        )

        process_instances_case_master_join_condition =(
            process_instances.cust_code == case_master.cust_code
        )

        
        process_instance_steps_process_definition_steps_join_condition = (
            process_instance_steps.pdsp_code == process_definition_steps.pdsp_code
        )

        process_instance_steps_lst_of_val_join_condition = (
            process_instance_steps.prst_status == lst_of_val.lov_low
        ) & (lst_of_val.lovt_code == 623)

        process_instance_steps_process_instance_mapping_activities_join_condition = (
            process_instance_steps.prst_code
            == process_instance_mapping_activities.prst_code
        )

        process_instance_participants_distinct_process_instance_mapping_activities_join_condition = (
            process_instance_participants_distinct.case_code
            == process_instance_mapping_activities.case_code
        )

        process_instance_participants_distinct_case_master_join_condition = (
            process_instance_participants_distinct.case_code == case_master.case_code
        )

        portfolio_case_master_join_condition = (
            portfolio.vendor_code_qualco == case_master.vend_code
        )

        customer_case_master_join_condition = (
            customer.cust_code == case_master.cust_code
        )

        # Join
        joined_df = (
            process_instance_steps.join(
            process_instances,
            process_instance_steps_process_instances_join_condition,
            "inner"
            )
            #.join(
            #case_master,
            #process_instance_participants_distinct_case_master_join_condition,
            #"left"
            #)
            .join(
                case_master,
                process_instances_case_master_join_condition,
                "left"
            )
            .join(
                process_definition_steps,
                process_instance_steps_process_definition_steps_join_condition,
                "left",
            )
            .join(
                lst_of_val,
                process_instance_steps_lst_of_val_join_condition,
                "left",
            )
            .join(
                process_instance_mapping_activities,
                process_instance_steps_process_instance_mapping_activities_join_condition,
                "left",
            )
            .join(
                process_instance_participants_distinct,
                process_instance_participants_distinct_process_instance_mapping_activities_join_condition,
                "left",
            )
            .join(
                portfolio,
                portfolio_case_master_join_condition,
                "left",
            )
            .join(
                customer,
                customer_case_master_join_condition,
                "left",
            )
        )

        # ====================== HashKey ======================

        joined_df_primary_key_fields = [
            process_instance_steps.prst_code,
            process_instance_participants.case_code,
            process_instance_mapping_activities.cac_code,
            lit("qc"),
        ]

        joined_df = (
            joined_df.withColumn(
                "LegalProcessStepId",
                sha2(concat_ws(";", *joined_df_primary_key_fields), 256),
            )
            .withColumn(
                "LegalProcessStepKey",
                concat_ws(
                    ":",
                    process_instance_steps.prst_code.cast("string"),
                    coalesce(
                        process_instance_mapping_activities.cac_code.cast("string"),
                        lit("NA"),
                    ),
                    coalesce(
                        process_instance_mapping_activities.case_code.cast("string"),
                        lit("NA"),
                    ),
                ),
            )
            .withColumn(
                "LegalStageKey",
                coalesce(
                    process_instance_steps.pris_code, process_instance_steps.pips_code
                ),
            )
        )

        # ================== Selection of Final Columns ==================

        result_df = joined_df.select(
            joined_df.LegalProcessStepId.cast("string"),
            lit("qc").cast("String").alias("SourceSystem"),
            case_master.vend_code.cast("String").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("String").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("String").alias("BatchId"),
            joined_df.LegalProcessStepKey.cast("string"),
            process_instance_steps.prin_code.cast("int").alias("LegalProcessKey"),
            joined_df.LegalStageKey.cast("int"),
            process_instance_steps.prst_code.cast("int").alias("LegalStepKey"),
            process_instance_mapping_activities.cac_code.cast("int").alias(
                "LegalActionKey"
            ),
            case_master.old_case_code.cast("string").alias("ExposureKey"),
            case_master.cust_code.cast("int").alias("PartyKey"),
            customer.CDI.alias("CDI"),
            process_instance_participants.case_code.cast("int").alias("CaseCode"),
            process_instance_steps.prst_created_by.cast("string").alias(
                "LegalStepCreatedBy"
            ),
            process_instance_steps.prst_created.cast("timestamp").alias(
                "LegalStepCreatedDate"
            ),
            process_instance_steps.prst_start_date.cast("string").alias(
                "LegalStepStartDate"
            ),
            process_instance_steps.prst_completion_date.cast("string").alias(
                "LegalStepEndDate"
            ),
            lit(None).cast("string").alias("LegalStepSystemEndDate"),
            process_instance_steps.pdsp_code.cast("int").alias(
                "LegalStepDefinitionCode"
            ),
            process_definition_steps.pdsp_name.cast("string").alias(
                "LegalStepDefinition"
            ),
            process_instance_steps.prst_status.cast("int").alias("LegalStepStatusCode"),
            lst_of_val.lov_desc.cast("string").alias("LegalStepStatus"),
            process_instance_steps.prst_comments.cast("string").alias(
                "LegalStepComment"
            ),
            joined_df.businessdate.alias("BusinessDate"),
        )

        # ================== Append ==================

        result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_legal_process_step_qc(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)